# モデルのパラメータ数とメモリ使用量を調べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_transformers/model_size.ipynb)

NLP クックブック 12-09 ／ Transformers の基本操作

このモデルは Colab の GPU (T4 は 16GB) に載るのか、パラメータ数とメモリ消費の目安を知りたい。

In [1]:
import torch
from transformers import AutoConfig, AutoModel

for name in ["intfloat/multilingual-e5-small", "bert-base-uncased", "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"]:
    model = AutoModel.from_pretrained(name)
    n = sum(p.numel() for p in model.parameters())
    emb = model.get_input_embeddings().weight.numel()
    print(f"{name:46} {n / 1e6:7.1f}M パラメータ (うち埋め込み {emb / n:.0%})  float32 で {model.get_memory_footprint() / 1e9:.2f} GB")

intfloat/multilingual-e5-small                   117.7M パラメータ (うち埋め込み 82%)  float32 で 0.47 GB
bert-base-uncased                                109.5M パラメータ (うち埋め込み 21%)  float32 で 0.44 GB
sbintuitions/sarashina2.2-0.5b-instruct-v0.1     662.0M パラメータ (うち埋め込み 20%)  float32 で 1.32 GB


In [2]:
# モデルを読み込まずに設定だけ見る
cfg = AutoConfig.from_pretrained("sbintuitions/sarashina2.2-0.5b-instruct-v0.1")
print({k: getattr(cfg, k) for k in ["hidden_size", "num_hidden_layers", "num_attention_heads", "vocab_size", "max_position_embeddings"]})

# パラメータ数からメモリの目安を計算する
def memory_gb(n_params, bits):
    return n_params * bits / 8 / 1e9

for label, n in [("0.5B", 0.5e9), ("7B", 7e9), ("70B", 70e9)]:
    print(f"{label:5} float32 {memory_gb(n, 32):6.1f} GB / float16 {memory_gb(n, 16):6.1f} GB / 8bit {memory_gb(n, 8):5.1f} GB / 4bit {memory_gb(n, 4):5.1f} GB")

{'hidden_size': 1280, 'num_hidden_layers': 24, 'num_attention_heads': 16, 'vocab_size': 102400, 'max_position_embeddings': 8192}
0.5B  float32    2.0 GB / float16    1.0 GB / 8bit   0.5 GB / 4bit   0.2 GB
7B    float32   28.0 GB / float16   14.0 GB / 8bit   7.0 GB / 4bit   3.5 GB
70B   float32  280.0 GB / float16  140.0 GB / 8bit  70.0 GB / 4bit  35.0 GB


- 推論時のメモリ ≒ パラメータ数 × 1 パラメータのバイト数 + α (活性化や KV キャッシュ)。学習時は勾配とオプティマイザの状態で、さらに 3〜4 倍必要。
- T4 (16GB) なら、float16 で 7B 級がぎりぎり、4bit 量子化なら 13B 級まで載る。
- `torch.cuda.memory_allocated()` や `!nvidia-smi` で実際の使用量を確認できる。